# Chapter 11 Trace: 2nd Derivative Spectral Modeling

`SpectralAnalysisPractice/SpeAna11_1-6_practicaluse.ipynb` の実行内容を、手元の `data/train.csv` / `data/test.csv` に合わせてトレースする notebook です。

今回の方針:

- 評価の主軸は Savitzky-Golay 2次微分スペクトル
- 含水率 `含水率` の回帰を中心に扱う
- 樹種名はモデル入力に使わず、通常CVを主評価にする。GroupKFold by 樹種は必要時のストレステストとしてだけ確認する
- 第11章の最後にある PCA + SVM 分類は、モデル構築には使わず、スペクトル構造を理解するための任意診断として扱う

第11章の対応:

- 11.1 アウトライヤーの検出と除去
- 11.2 各モジュールでの標準化自由度
- 11.3 PLSウェイトローディング，ローディング，回帰係数
- 11.4 ウェイトローディングと寄与率
- 11.5 GridSearchCVによるクロスバリデーション
- 11.6 パイプラインを用いたモデルの最適化

## このノートでの樹種の扱い

このノート以降、樹種名・樹種番号はモデル入力として使いません。目的は、樹種が既知か不明かに依存せず、スペクトルから含水率を推定できるモデルを作ることです。

樹種情報を使う場合は、外れ値確認や失敗原因の診断など、結果を理解するためのメタデータに限定します。樹種別モデル、樹種分類を前提にしたモデル、樹種名による予測補正は基本方針から外します。


## ライブラリをまとめてインポート

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns

from scipy.signal import savgol_filter

from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC, SVR

sns.set_theme(style="whitegrid", context="notebook")

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Hiragino Sans", "AppleGothic", "Apple SD Gothic Neo", "Yu Gothic", "Meiryo", "Noto Sans CJK JP"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        print(f"Matplotlib font: {font_name}")
        break

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMIT_PATH = DATA_DIR / "sample_submit.csv"

ENCODING = "cp932"
ID_COLS = ["sample number", "species number", "樹種"]
TARGET_COL = "含水率"
RANDOM_STATE = 42

## コード10.1 / 10.2 相当: CSV読み込みとラベル作成

第11章は第10章で作った `spectra` と `prop` を前提に始まります。ここでも同じ名前の変数を作ります。

In [ ]:
train = pd.read_csv(TRAIN_PATH, encoding=ENCODING)
test = pd.read_csv(TEST_PATH, encoding=ENCODING)

spectral_cols = [c for c in train.columns if c not in ID_COLS + [TARGET_COL]]
wave = np.array([float(c) for c in spectral_cols])

spectra = train[spectral_cols].astype(float)
test_spectra = test[spectral_cols].astype(float)
prop = train[["sample number", "species number", "樹種", TARGET_COL]].copy()
prop = prop.rename(columns={TARGET_COL: "mc"})
prop["label"] = prop["species number"].astype(int)

y = prop["mc"].astype(float)
groups = prop["樹種"]

print(f"spectra: {spectra.shape}")
print(f"test_spectra: {test_spectra.shape}")
print(f"wave range: {wave.max():.2f} - {wave.min():.2f} cm^-1")
display(prop.head())
display(prop["樹種"].value_counts().to_frame("count"))

## 2次微分スペクトルを作成

第11章のコード11.13以降では微分スペクトルを比較します。今回は以後の主データとして2次微分スペクトルを使います。

In [ ]:
def odd_window(window_length, n_points):
    window_length = int(window_length)
    if window_length % 2 == 0:
        window_length += 1
    max_window = n_points if n_points % 2 == 1 else n_points - 1
    return min(window_length, max_window)


def differentiate_spectra(X, order=2, window_length=21, polyorder=2):
    window_length = odd_window(window_length, np.asarray(X).shape[1])
    return savgol_filter(np.asarray(X, dtype=float), window_length=window_length, polyorder=polyorder, deriv=order, axis=1)

DERIV_ORDER = 2
DERIV_WINDOW = 21
DERIV_POLYORDER = 2

X_deriv = differentiate_spectra(spectra, order=DERIV_ORDER, window_length=DERIV_WINDOW, polyorder=DERIV_POLYORDER)
X_deriv = pd.DataFrame(X_deriv, index=spectra.index, columns=spectral_cols)

test_deriv = differentiate_spectra(test_spectra, order=DERIV_ORDER, window_length=DERIV_WINDOW, polyorder=DERIV_POLYORDER)
test_deriv = pd.DataFrame(test_deriv, index=test_spectra.index, columns=spectral_cols)

print(X_deriv.shape, test_deriv.shape)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(wave, spectra.iloc[:80].T, alpha=0.18, linewidth=0.8)
axes[0].set_title("Raw spectra")
axes[0].set_ylabel("Intensity")
axes[1].plot(wave, X_deriv.iloc[:80].T, alpha=0.18, linewidth=0.8)
axes[1].set_title(f"2nd derivative spectra (window={DERIV_WINDOW}, polyorder={DERIV_POLYORDER})")
axes[1].set_xlabel("Wavenumber (cm^-1)")
axes[1].set_ylabel("2nd derivative")
for ax in axes:
    ax.invert_xaxis()
plt.tight_layout()

## 11.1 アウトライヤーの検出と除去

教材のコード11.1/11.2に対応します。PCAスコア空間と目的変数それぞれに IsolationForest を使い、外れ値候補を確認します。

In [ ]:
def remove_outliers_and_plot(spec, prop_data, contamination=0.05, wave_values=None):
    spec_df = pd.DataFrame(spec).astype(float)
    prop_series = pd.Series(prop_data).astype(float).reset_index(drop=True)

    pca = PCA(n_components=4, random_state=RANDOM_STATE)
    spectra_pca = pca.fit_transform(StandardScaler().fit_transform(spec_df))

    model_spec = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
    predict_spec = model_spec.fit_predict(spectra_pca)

    model_prop = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
    predict_prop = model_prop.fit_predict(prop_series.values.reshape(-1, 1))

    outlier_idx_spec = np.where(predict_spec == -1)[0]
    outlier_idx_prop = np.where(predict_prop == -1)[0]
    outlier_idx = np.union1d(outlier_idx_spec, outlier_idx_prop)

    print("スペクトルからのアウトライヤー数:", len(outlier_idx_spec))
    print("目的変数からのアウトライヤー数:", len(outlier_idx_prop))
    print("統合アウトライヤー数:", len(outlier_idx))

    keep_mask = np.ones(len(spec_df), dtype=bool)
    keep_mask[outlier_idx] = False
    spectra_isolated = spec_df.iloc[keep_mask].copy()
    prop_isolated = prop_series.iloc[keep_mask].copy()

    fig, axes = plt.subplots(3, 1, figsize=(9, 10))
    axes[0].scatter(range(len(prop_series)), prop_series, color="black", s=12, label="Normal")
    axes[0].scatter(outlier_idx, prop_series.iloc[outlier_idx], color="red", s=24, label="Outlier")
    axes[0].set_xlabel("Sample Number")
    axes[0].set_ylabel("Property")
    axes[0].legend()

    axes[1].scatter(spectra_pca[:, 0], spectra_pca[:, 1], color="black", s=12, label="Normal")
    axes[1].scatter(spectra_pca[outlier_idx, 0], spectra_pca[outlier_idx, 1], color="red", s=24, label="Outlier")
    axes[1].set_xlabel("PC1 Score")
    axes[1].set_ylabel("PC2 Score")
    axes[1].legend()

    axes[2].scatter(spectra_pca[:, 2], spectra_pca[:, 3], color="black", s=12, label="Normal")
    axes[2].scatter(spectra_pca[outlier_idx, 2], spectra_pca[outlier_idx, 3], color="red", s=24, label="Outlier")
    axes[2].set_xlabel("PC3 Score")
    axes[2].set_ylabel("PC4 Score")
    axes[2].legend()
    plt.tight_layout()
    plt.show()

    x_axis = np.arange(spec_df.shape[1]) if wave_values is None else np.asarray(wave_values, dtype=float)
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for i, ax in enumerate(axes.ravel()):
        ax.plot(x_axis, pca.components_[i], color="black")
        ax.set_title(f"PC{i + 1} Loadings")
        ax.set_xlabel("Wavenumber (cm^-1)" if wave_values is not None else "Wavelength Index")
        ax.set_ylabel("Loading")
        if wave_values is not None:
            ax.invert_xaxis()
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for i, ax in enumerate(axes.ravel()):
        ax.scatter(spectra_pca[:, i], prop_series, color="black", s=12, label="Normal")
        ax.scatter(spectra_pca[outlier_idx, i], prop_series.iloc[outlier_idx], color="red", s=24, label="Outlier")
        ax.set_title(f"PC{i + 1} score vs prop")
        ax.set_xlabel("score")
        ax.set_ylabel("prop")
    plt.tight_layout()
    plt.show()

    outlier_table = prop.iloc[outlier_idx][["sample number", "species number", "樹種", "mc"]].copy()
    outlier_table["outlier_source"] = [
        ",".join([name for name, idxs in [("spectra", outlier_idx_spec), ("property", outlier_idx_prop)] if i in set(idxs)])
        for i in outlier_idx
    ]
    display(outlier_table.head(30))
    return spectra_isolated, prop_isolated, outlier_idx, outlier_table

In [ ]:
spectra_isolated, prop_isolated, outlier_idx, outlier_table = remove_outliers_and_plot(
    X_deriv,
    prop["mc"],
    contamination=0.05,
    wave_values=wave,
)

## 11.2 各モジュールでの標準化自由度

教材のコード11.3に対応します。`pandas`, `numpy`, `StandardScaler` の標準化結果が同じになることを確認します。

In [ ]:
spectra_std_pd = (X_deriv - X_deriv.mean()) / X_deriv.std(ddof=0)

spectra_array = X_deriv.values
spectra_std_np = (spectra_array - np.mean(spectra_array, axis=0)) / np.std(spectra_array, axis=0, ddof=0)

scaler = StandardScaler()
spectra_std_scaler = scaler.fit_transform(spectra_array)

check_df = pd.DataFrame({
    "pandas": spectra_std_pd.iloc[0, :8].values,
    "numpy": spectra_std_np[0, :8],
    "StandardScaler": spectra_std_scaler[0, :8],
}, index=spectral_cols[:8])

display(check_df)
print("pandas vs numpy allclose:", np.allclose(spectra_std_pd.values, spectra_std_np))
print("numpy vs StandardScaler allclose:", np.allclose(spectra_std_np, spectra_std_scaler))

## 11.3 PLSウェイトローディング，ローディング，回帰係数

教材のコード11.4/11.5に対応します。ここでは2次微分スペクトルで PLS を学習し、ウェイトローディング・ローディング・回帰係数を確認します。

In [ ]:
PLS_COMPONENTS_FOR_VIEW = 5
pls = PLSRegression(n_components=PLS_COMPONENTS_FOR_VIEW, scale=True)
pls.fit(X_deriv, y)

weight = pls.x_weights_
loading = pls.x_loadings_
coef = pls.coef_.ravel()

fig, axes = plt.subplots(PLS_COMPONENTS_FOR_VIEW, 1, figsize=(13, 11), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(wave, weight[:, i], "r-", linewidth=1.4, label="Weight loading")
    ax.plot(wave, loading[:, i], "k-", linewidth=1.0, label="Loading")
    ax.set_title(f"Component {i + 1}")
    ax.set_ylabel("value")
    ax.invert_xaxis()
axes[-1].set_xlabel("Wavenumber (cm^-1)")
axes[0].legend()
plt.tight_layout()

plt.figure(figsize=(13, 4))
plt.plot(wave, coef, color="purple")
plt.gca().invert_xaxis()
plt.xlabel("Wavenumber (cm^-1)")
plt.ylabel("Regression coefficient")
plt.title("PLS regression coefficients")
plt.tight_layout()

In [ ]:
pred_method = pls.predict(X_deriv).ravel()
print("PLS predict method:", np.round(pred_method[:6], 3))
print("Measured          :", np.round(y.iloc[:6].values, 3))

plt.figure(figsize=(5, 5))
plt.scatter(y, pred_method, s=18, alpha=0.75)
lims = [min(y.min(), pred_method.min()), max(y.max(), pred_method.max())]
plt.plot(lims, lims, "k-", lw=1)
plt.xlabel("Measured")
plt.ylabel("Predicted")
plt.title("PLS calibration: measured vs predicted")
plt.tight_layout()

## 11.4 ウェイトローディングと寄与率

教材のコード11.6/11.7/11.8/11.9に対応します。PLS成分数を増やしながら、train R2・CV R2・寄与率を確認します。

In [ ]:
def pls_cv_check(spec, prop_data, n_folds=5, max_components=10, scaleset=True, groups=None):
    spec_df = pd.DataFrame(spec).astype(float)
    prop_series = pd.Series(prop_data).astype(float)

    scores_cv = []
    scores_train = []
    explained_var_spec = []
    explained_var_prop = []
    coefficients = []
    pred = []

    if scaleset:
        spec_normalized = (spec_df - spec_df.mean()) / spec_df.std(ddof=1)
    else:
        spec_normalized = spec_df - spec_df.mean()

    if groups is None:
        cv = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
        cv_splitter = cv
    else:
        cv = GroupKFold(n_splits=n_folds)
        cv_splitter = cv.split(spec_df, prop_series, groups=groups)

    for n in range(1, max_components + 1):
        pls_model = PLSRegression(n_components=n, scale=scaleset)
        if groups is None:
            score_cv = cross_val_score(pls_model, spec_df, prop_series, cv=cv, scoring="r2").mean()
        else:
            score_cv = cross_val_score(pls_model, spec_df, prop_series, cv=cv, groups=groups, scoring="r2").mean()

        pls_model.fit(spec_df, prop_series)
        score_train = pls_model.score(spec_df, prop_series)
        transformed_spec = pls_model.transform(spec_df)
        predicted = pls_model.predict(spec_df).ravel()

        scores_cv.append(score_cv)
        scores_train.append(score_train)
        explained_var_spec.append(np.var(transformed_spec, axis=0).sum() / np.var(spec_normalized, axis=0).sum())
        explained_var_prop.append(np.var(predicted, axis=0).sum() / np.var(prop_series, axis=0))
        coefficients.append(pls_model.coef_)
        pred.append(predicted)

    optimal_n = int(np.argmax(scores_cv) + 1)
    final_pls = PLSRegression(n_components=optimal_n, scale=scaleset).fit(spec_df, prop_series)
    print("最適な主成分数は", optimal_n)

    results = {
        "optimal_n_components": optimal_n,
        "pred_value": pred,
        "r2_scores_train": scores_train,
        "r2_scores_cv": scores_cv,
        "explained_variances_spec": explained_var_spec,
        "explained_variances_prop": explained_var_prop,
        "loadings": final_pls.x_loadings_,
        "weights": final_pls.x_weights_,
        "regression_coefficients": coefficients,
    }

    fig, axes = plt.subplots(3, 2, figsize=(14, 11))
    axes = axes.ravel()
    axes[0].plot(wave, spec_normalized.T, color="k", linewidth=0.25, alpha=0.08)
    axes[0].invert_xaxis()
    axes[0].set_xlabel("Wavenumber (cm^-1)")
    axes[0].set_ylabel("Normalized 2nd derivative")
    axes[0].set_title("Normalized spectra")

    axes[2].plot(range(1, max_components + 1), scores_train, "k-o", label="Train")
    axes[2].plot(range(1, max_components + 1), scores_cv, "r-o", label="CV")
    axes[2].set_xlabel("Number of PLS Components")
    axes[2].set_ylabel("R2 Score")
    axes[2].set_title("PLS Components vs R2 Score")
    axes[2].legend()

    axes[4].plot(range(1, max_components + 1), explained_var_spec, "m-o", label="Spectra")
    axes[4].plot(range(1, max_components + 1), explained_var_prop, "c-o", label="Property")
    axes[4].set_xlabel("Number of PLS Components")
    axes[4].set_ylabel("Explained Variance")
    axes[4].set_title("PLS Components vs Explained Variance")
    axes[4].legend()

    max_weight_components = min(max_components, final_pls.x_weights_.shape[1])
    for ax, start, end, title in [(axes[1], 0, 3, "Weightloadings (Components 1-3)"), (axes[3], 3, 6, "Weightloadings (Components 4-6)"), (axes[5], 6, 10, "Weightloadings (Components 7-10)")]:
        for i in range(start, min(end, max_weight_components)):
            ax.plot(wave, final_pls.x_weights_[:, i], label=f"Component {i + 1}")
        ax.invert_xaxis()
        ax.set_xlabel("Wavenumber (cm^-1)")
        ax.set_ylabel("Weightloading")
        ax.set_title(title)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend()

    plt.tight_layout()
    plt.show()
    return results

In [ ]:
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_deriv,
    y,
    groups,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=groups,
)

result_random_cv = pls_cv_check(X_train, y_train, n_folds=5, max_components=10, scaleset=True)
result_group_cv = pls_cv_check(X_train, y_train, n_folds=5, max_components=10, scaleset=True, groups=groups_train)

cv_compare = pd.DataFrame({
    "n_components": range(1, 11),
    "random_cv_r2": result_random_cv["r2_scores_cv"],
    "group_cv_r2": result_group_cv["r2_scores_cv"],
    "train_r2": result_random_cv["r2_scores_train"],
})
display(cv_compare)

In [ ]:
optimal_components = result_group_cv["optimal_n_components"]
pls = PLSRegression(n_components=optimal_components, scale=True)
pls.fit(X_train, y_train)
y_pred = pls.predict(X_test).ravel()

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
print(f"PLS components: {optimal_components}")
print(f"R2  : {r2:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE : {mae:.3f}")

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, c="m", label="opt", s=24, alpha=0.75)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, "k-", lw=1)
plt.xlabel("Measured")
plt.ylabel("Predicted")
plt.title("Measured vs Predicted: PLS on 2nd derivative spectra")
plt.legend()
plt.tight_layout()

## コード11.10 / 11.11 pickleによるモデル保存と読み込み

In [ ]:
model_path = OUTPUT_DIR / "pls_2nd_derivative_model.pkl"
with open(model_path, "wb") as file:
    pickle.dump(pls, file)

with open(model_path, "rb") as file:
    loaded_pls = pickle.load(file)

loaded_y_pred = loaded_pls.predict(X_test).ravel()
print("saved model:", model_path)
print("loaded prediction allclose:", np.allclose(y_pred, loaded_y_pred))

## 11.5 GridSearchCVによるクロスバリデーション

教材のコード11.12に対応します。PLS と SVR を2次微分スペクトルで比較します。

In [ ]:
pls_grid = GridSearchCV(
    PLSRegression(scale=True),
    param_grid={"n_components": list([1, 2, 3, 5, 8, 10])},
    cv=5,
    scoring="r2",
)
pls_grid.fit(X_train, y_train)
print(f"Best PLS components: {pls_grid.best_params_['n_components']}")

svr_grid = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=10, random_state=RANDOM_STATE)),
        ("svr", SVR(kernel="rbf")),
    ]),
    param_grid={
        "svr__C": [10],
        "svr__gamma": ["scale"],
    },
    cv=3,
    scoring="r2",
)
svr_grid.fit(X_train, y_train)
print(f"Best SVR parameters: {svr_grid.best_params_}")

models_for_eval = {
    "PLS": pls_grid.best_estimator_,
    "SVR": svr_grid.best_estimator_,
}

for name, model in models_for_eval.items():
    pred = model.predict(X_test)
    pred = np.asarray(pred).ravel()
    print(f"{name} R2: {r2_score(y_test, pred):.3f}, RMSE: {mean_squared_error(y_test, pred) ** 0.5:.3f}, MAE: {mean_absolute_error(y_test, pred):.3f}")

    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, pred, edgecolors="black", alpha=0.75)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    plt.plot(lims, lims, "k-", lw=1)
    plt.xlabel("Measured")
    plt.ylabel("Predicted")
    plt.title(f"{name}: Measured vs Predicted")
    plt.tight_layout()
    plt.show()

## 11.6 パイプラインを用いたモデルの最適化

### コード11.13 微分スペクトルを用いたPLS回帰モデルの最適化

教材では1次微分と2次微分を比較します。ここでも比較しますが、最終方針は2次微分を主軸にします。

In [ ]:
def optimize_pls(X, y_data, max_components=12, cv=5):
    max_components = min(max_components, X.shape[0] - 1, X.shape[1])
    pls_params = {"n_components": range(1, max_components + 1)}
    pls_grid = GridSearchCV(PLSRegression(scale=True), param_grid=pls_params, cv=cv, scoring="r2")
    pls_grid.fit(X, y_data)
    return pls_grid.best_estimator_, pls_grid.best_score_

windows = [9, 21, 41]
X_train_raw = X_train.values
X_train_1st_derivative = [differentiate_spectra(X_train_raw, order=1, window_length=w) for w in windows]
X_train_2nd_derivative = [differentiate_spectra(X_train_raw, order=2, window_length=w) for w in windows]

models = [optimize_pls(X_train_raw, y_train)]
models += [optimize_pls(X, y_train) for X in X_train_1st_derivative]
models += [optimize_pls(X, y_train) for X in X_train_2nd_derivative]

derivative_results = pd.DataFrame({
    "Model": ["Original"] + [f"1st Derivative (window={w})" for w in windows] + [f"2nd Derivative (window={w})" for w in windows],
    "Optimal Components": [model[0].n_components for model in models],
    "R2 (CV)": [round(model[1], 3) for model in models],
})
display(derivative_results)

### コード11.14 スムージングポイントの数による微分スペクトルの変化

In [ ]:
fig, axes = plt.subplots(len(windows), 2, figsize=(12, 2.4 * len(windows)), sharex=True)
for i, (w, X_1st, X_2nd) in enumerate(zip(windows, X_train_1st_derivative, X_train_2nd_derivative)):
    sample_idx = np.linspace(0, X_1st.shape[0] - 1, min(80, X_1st.shape[0])).astype(int)
    axes[i, 0].plot(wave, X_1st[sample_idx].T, color="blue", alpha=0.15, linewidth=0.7)
    axes[i, 0].set_title(f"1st Derivative (window={w})")
    axes[i, 1].plot(wave, X_2nd[sample_idx].T, color="red", alpha=0.15, linewidth=0.7)
    axes[i, 1].set_title(f"2nd Derivative (window={w})")
    for ax in axes[i]:
        ax.invert_xaxis()
        ax.set_ylabel("value")
axes[-1, 0].set_xlabel("Wavenumber (cm^-1)")
axes[-1, 1].set_xlabel("Wavenumber (cm^-1)")
plt.tight_layout()

### コード11.15 / 11.16 / 11.17 パイプラインとハイパーパラメータチューニング

教材では smoothing + PCA + LinearRegression の Pipeline を使います。ここでは Savitzky-Golay 2次微分を Pipeline の先頭に入れます。

In [ ]:
def sg_transform(X, window_length=21, polyorder=2, deriv=2):
    return differentiate_spectra(X, order=deriv, window_length=window_length, polyorder=polyorder)

pipeline = Pipeline([
    ("derivative", FunctionTransformer(sg_transform, validate=False, kw_args={"window_length": 21, "polyorder": 2, "deriv": 2})),
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=RANDOM_STATE)),
    ("regression", LinearRegression()),
])

param_grid = {
    "derivative__kw_args": [
        {"window_length": w, "polyorder": 2, "deriv": 2}
        for w in [13, 21]
    ],
    "pca__n_components": [3, 5, 8],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2", n_jobs=None)
grid_search.fit(spectra.loc[X_train.index], y_train)
print("Best parameters:", grid_search.best_params_)
print(f"Best CV R2: {grid_search.best_score_:.3f}")

predicted = grid_search.predict(spectra.loc[X_test.index])
print(f"PCR pipeline R2: {r2_score(y_test, predicted):.3f}, RMSE: {mean_squared_error(y_test, predicted) ** 0.5:.3f}, MAE: {mean_absolute_error(y_test, predicted):.3f}")

### コード11.18 さまざまな回帰アルゴリズムを使用したパイプラインの定義と最適化

探索を重くしすぎないため、各モデルのグリッドは小さめにしています。必要に応じて広げます。

In [ ]:
RUN_EXTENDED_MODELS = False

pipelines = {
    "PCR": Pipeline([("scaler", StandardScaler()), ("pca", PCA(random_state=RANDOM_STATE)), ("regressor", LinearRegression())]),
    "PLS": Pipeline([("regressor", PLSRegression(scale=True))]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("regressor", Ridge())]),
}

param_grids = {
    "PCR": {"pca__n_components": [3, 8]},
    "PLS": {"regressor__n_components": [3, 8]},
    "Ridge": {"regressor__alpha": [1.0, 10.0]},
}

if RUN_EXTENDED_MODELS:
    pipelines.update({
        "SVR": Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=10, random_state=RANDOM_STATE)), ("regressor", SVR())]),
        "RandomForest": Pipeline([("regressor", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
        "GradientBoosting": Pipeline([("regressor", GradientBoostingRegressor(random_state=RANDOM_STATE))]),
    })
    param_grids.update({
        "SVR": {"regressor__C": [10], "regressor__gamma": ["scale"]},
        "RandomForest": {"regressor__n_estimators": [40], "regressor__max_depth": [8]},
        "GradientBoosting": {"regressor__n_estimators": [60], "regressor__learning_rate": [0.05]},
    })

model_results = []
best_estimators = {}
for name, pipe in pipelines.items():
    grid = GridSearchCV(pipe, param_grids[name], cv=3, scoring="r2")
    grid.fit(X_train, y_train)
    best_estimators[name] = grid.best_estimator_
    pred = np.asarray(grid.predict(X_test)).ravel()
    model_results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV R2": grid.best_score_,
        "Test R2": r2_score(y_test, pred),
        "Test RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "Test MAE": mean_absolute_error(y_test, pred),
    })

model_results_df = pd.DataFrame(model_results).sort_values("Test RMSE")
display(model_results_df)


## 任意診断: GroupKFold by 樹種で外挿ストレスを確認

第11章の教材は通常CV中心です。この節では、樹種名をモデルに使わない前提のまま、樹種単位で分布が変わった場合のストレステストとしてGroupKFoldも任意で確認します。主評価は通常CVです。

In [ ]:
group_cv = GroupKFold(n_splits=5)
group_model_defs = {
    "PLS": Pipeline([("regressor", PLSRegression(n_components=optimal_components, scale=True))]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("regressor", Ridge(alpha=10.0))]),
    "SVR": svr_grid.best_estimator_,
}

group_rows = []
for name, model in group_model_defs.items():
    pred = cross_val_predict(model, X_deriv, y, cv=group_cv, groups=groups)
    group_rows.append({
        "Model": name,
        "Group stress R2": r2_score(y, pred),
        "Group stress RMSE": mean_squared_error(y, pred) ** 0.5,
        "Group stress MAE": mean_absolute_error(y, pred),
    })

group_results_df = pd.DataFrame(group_rows).sort_values("Group stress RMSE")
display(group_results_df)

plt.figure(figsize=(6, 6))
best_group_name = group_results_df.iloc[0]["Model"]
best_group_model = group_model_defs[best_group_name]
best_group_pred = cross_val_predict(best_group_model, X_deriv, y, cv=group_cv, groups=groups)
plt.scatter(y, best_group_pred, s=18, alpha=0.7)
lims = [min(y.min(), best_group_pred.min()), max(y.max(), best_group_pred.max())]
plt.plot(lims, lims, "k-", lw=1)
plt.xlabel("Measured")
plt.ylabel("Predicted")
plt.title(f"Optional group stress test: {best_group_name}")
plt.tight_layout()

## コード11.19 PCAとSVMを用いたクロスバリデーションとハイパーパラメータ最適化

ここからは回帰ではなく、スペクトル構造を理解するための任意診断です。この分類器は含水率モデルの前段にも、test予測にも使いません。

In [ ]:
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_deriv,
    prop[["mc", "label", "樹種"]],
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=prop["label"],
)

classification_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=RANDOM_STATE)),
    ("svm", SVC(probability=True, random_state=RANDOM_STATE)),
])

classification_param_grid = {
    "pca__n_components": [3, 5],
    "svm__C": [10],
    "svm__gamma": ["scale"],
}

classification_grid = GridSearchCV(
    classification_pipeline,
    classification_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="accuracy",
)
classification_grid.fit(X_cls_train, y_cls_train["label"])

print("Best parameters:")
print(classification_grid.best_params_)
print("Best cross-validation score: {:.3f}".format(classification_grid.best_score_))

y_cls_pred = classification_grid.predict(X_cls_test)
print(classification_report(y_cls_test["label"], y_cls_pred, zero_division=0))

## コード11.20 最適なパラメータでPCAとSV分類を設定

In [ ]:
pca = PCA(n_components=3, random_state=RANDOM_STATE)
svm = SVC(C=100, gamma=0.1, probability=True, random_state=RANDOM_STATE)

X_train_pca = pca.fit_transform(StandardScaler().fit_transform(X_cls_train))
mean_spectrum = X_cls_train.mean(axis=0)

fig, axes = plt.subplots(3, 2, figsize=(13, 9))
for i in range(3):
    axes[i, 0].scatter(y_cls_train["mc"], X_train_pca[:, i], c=y_cls_train["label"], cmap="tab20", s=18)
    axes[i, 0].set_xlabel("mc")
    axes[i, 0].set_ylabel(f"PC{i + 1}")

for i in range(3):
    axes[i, 1].plot(wave, pca.components_[i], label=f"PC{i + 1}")
    axes[i, 1].plot(wave, mean_spectrum.values, color="r", alpha=0.6, label="Mean Spectrum")
    axes[i, 1].invert_xaxis()
    axes[i, 1].set_xlabel("Wavenumber (cm^-1)")
    axes[i, 1].set_ylabel("Loading")
    axes[i, 1].legend()
plt.tight_layout()

## コード11.21 PCAを用いたSV分類

In [ ]:
scaler_cls = StandardScaler()
X_train_scaled = scaler_cls.fit_transform(X_cls_train)
X_test_scaled = scaler_cls.transform(X_cls_test)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

for pair, label in [([0, 1], "PC1 and PC2"), ([1, 2], "PC2 and PC3"), ([0, 2], "PC1 and PC3")]:
    svm.fit(X_train_pca[:, pair], y_cls_train["label"])
    y_pred_pair = svm.predict(X_test_pca[:, pair])
    print(f"Using {label}:")
    print(classification_report(y_cls_test["label"], y_pred_pair, zero_division=0))

## test.csv への予測準備

第11章のモデル評価後に、最終 test へ予測を出すための最小セルです。樹種名は使わず、スペクトルだけで学習したモデルを使います。

In [ ]:
final_model = Pipeline([
    ("regressor", PLSRegression(n_components=optimal_components, scale=True)),
])
final_model.fit(X_deriv, y)
test_pred = final_model.predict(test_deriv).ravel()

prediction_df = pd.DataFrame({
    "sample number": test["sample number"],
    "predicted_mc": test_pred,
})

display(prediction_df.head())
print(prediction_df["predicted_mc"].describe())

prediction_path = OUTPUT_DIR / "test_predictions_pls_2nd_derivative.csv"
prediction_df.to_csv(prediction_path, index=False)
print("saved:", prediction_path)

## 第11章トレース後の確認ポイント

- 2次微分スペクトルを主軸に、PLSの成分数・window_length・回帰モデルを比較した。
- 樹種名・樹種番号はモデル入力に使わない。
- 主評価はスペクトル単独モデルの通常CVとし、GroupKFoldは分布変化への任意ストレステストとして扱う。
- 次に改善するなら、`DERIV_WINDOW`、外れ値処理の有無、PLS/Ridge/SVR の通常CVスコアを軸に比較する。